# Inspect source files (data catalog)

This notebook loads the inventory produced by `src/catalog/build_data_catalog.py` and explores what lives under `data/raw/`.

**Tip:** Run Jupyter with the **project root** as the working directory (the folder that contains `outputs/` and `data/`). The code below searches upward from the current folder until it finds `outputs/tables/data_catalog.csv`.

## 1. Load the catalog CSV

We use **pandas** `read_csv` to read the table into a **DataFrame** (`df`). Each row is one file discovered under `data/raw/`.

In [17]:
from pathlib import Path

import pandas as pd


def find_project_root() -> Path:
    """Walk up from cwd until we find the catalog CSV (works from project root or notebooks/)."""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        catalog = folder / "outputs" / "tables" / "data_catalog.csv"
        if catalog.is_file():
            return folder
    raise FileNotFoundError(
        "Could not find outputs/tables/data_catalog.csv. "
        "Open the project folder as Jupyter's root or run: cd <project-root>"
    )


ROOT = find_project_root()
catalog_path = ROOT / "outputs" / "tables" / "data_catalog.csv"

df = pd.read_csv(catalog_path)
print(f"Loaded {len(df)} rows from {catalog_path}")

Loaded 91 rows from /Users/iagolarrondo/Library/CloudStorage/OneDrive-MassachusettsInstituteofTechnology/1. Coursework and academic/Y2 ('25-'26) - IAP, H3 & H4/15.S04 – Gen AI Lab/2. Assignments and deliverables/Manulife Project/outputs/tables/data_catalog.csv


## 2. Display the full catalog

Show every row. We temporarily raise pandas’ row limit so nothing is hidden (your catalog is small enough for this).

In [18]:
with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.width", None):
    display(df)

,relative_path,file_name,extension,top_level_group
0,.DS_Store,.DS_Store,NaN,other
1,ddl/.gitkeep,.gitkeep,NaN,ddl
2,ddl/Create_Table__INVESTIGAI_ICP_CAREGIVER_APP...,Create_Table__INVESTIGAI_ICP_CAREGIVER_APP_SES...,.sql,ddl
3,ddl/Create_Table__INVESTIGAI_ICP_WORK_BLOCKS.sql,Create_Table__INVESTIGAI_ICP_WORK_BLOCKS.sql,.sql,ddl
4,ddl/Create_Table__INVESTIGAI_PROVIDER_CLAIMS.sql,Create_Table__INVESTIGAI_PROVIDER_CLAIMS.sql,.sql,ddl
5,ddl/Create_Table__T_NORM_CARE_SESSION.sql,Create_Table__T_NORM_CARE_SESSION.sql,.sql,ddl
6,ddl/Create_Table__T_NORM_CARE_SESSION_EVENT.sql,Create_Table__T_NORM_CARE_SESSION_EVENT.sql,.sql,ddl
7,ddl/Create_Table__T_NORM_CHARGE.sql,Create_Table__T_NORM_CHARGE.sql,.sql,ddl
8,ddl/Create_Table__T_NORM_CLAIM.sql,Create_Table__T_NORM_CLAIM.sql,.sql,ddl
9,ddl/Create_Table__T_NORM_CLAIM_ELIGIBILITY_REV...,Create_Table__T_NORM_CLAIM_ELIGIBILITY_REVIEW_...,.sql,ddl


## 3. File counts by `top_level_group`

`top_level_group` comes from the first folder under `data/raw/` (`ddl`, `documentation`, `graph`) or `other` for files sitting directly in `raw/` (and anything else unexpected).

In [19]:
counts = df["top_level_group"].value_counts().sort_index()
display(counts)

top_level_group
ddl              39
documentation    48
graph             2
other             2
Name: count, dtype: int64

## 4. All `.sql` files (DDL)

Filter rows where the **extension** column is `.sql`. These are the Snowflake-style table/view scripts under `data/raw/ddl/`.

In [20]:
sql_files = df[df["extension"] == ".sql"].copy()
sql_files = sql_files.sort_values("relative_path")
display(sql_files[["relative_path", "file_name", "top_level_group"]])

,relative_path,file_name,top_level_group
2,ddl/Create_Table__INVESTIGAI_ICP_CAREGIVER_APP...,Create_Table__INVESTIGAI_ICP_CAREGIVER_APP_SES...,ddl
3,ddl/Create_Table__INVESTIGAI_ICP_WORK_BLOCKS.sql,Create_Table__INVESTIGAI_ICP_WORK_BLOCKS.sql,ddl
4,ddl/Create_Table__INVESTIGAI_PROVIDER_CLAIMS.sql,Create_Table__INVESTIGAI_PROVIDER_CLAIMS.sql,ddl
5,ddl/Create_Table__T_NORM_CARE_SESSION.sql,Create_Table__T_NORM_CARE_SESSION.sql,ddl
6,ddl/Create_Table__T_NORM_CARE_SESSION_EVENT.sql,Create_Table__T_NORM_CARE_SESSION_EVENT.sql,ddl
7,ddl/Create_Table__T_NORM_CHARGE.sql,Create_Table__T_NORM_CHARGE.sql,ddl
8,ddl/Create_Table__T_NORM_CLAIM.sql,Create_Table__T_NORM_CLAIM.sql,ddl
9,ddl/Create_Table__T_NORM_CLAIM_ELIGIBILITY_REV...,Create_Table__T_NORM_CLAIM_ELIGIBILITY_REVIEW_...,ddl
10,ddl/Create_Table__T_NORM_CLINICAL_REVIEW.sql,Create_Table__T_NORM_CLINICAL_REVIEW.sql,ddl
11,ddl/Create_Table__T_NORM_DIAGNOSIS.sql,Create_Table__T_NORM_DIAGNOSIS.sql,ddl


## 5. All `.txt` and `.md` files

Documentation snippets (`docs__*.txt`), the graph model write-up (`GRAPH_DATA_MODEL.md`), and the high-level `readme.md` overview.

In [21]:
text_like = df[df["extension"].isin([".txt", ".md"])].copy()
text_like = text_like.sort_values("relative_path")
display(text_like[["relative_path", "file_name", "top_level_group"]])

,relative_path,file_name,top_level_group
41,documentation/docs__COG_impairment.txt,docs__COG_impairment.txt,documentation
42,documentation/docs__DMB.txt,docs__DMB.txt,documentation
43,documentation/docs__HIPAA.txt,docs__HIPAA.txt,documentation
44,documentation/docs__INVESTIGAI_ICP_CAREGIVER_A...,docs__INVESTIGAI_ICP_CAREGIVER_APP_SESSIONS.txt,documentation
45,documentation/docs__POA.txt,docs__POA.txt,documentation
46,documentation/docs__T_NORM_CARE_SESSION.txt,docs__T_NORM_CARE_SESSION.txt,documentation
47,documentation/docs__T_NORM_CARE_SESSION_EVENT.txt,docs__T_NORM_CARE_SESSION_EVENT.txt,documentation
48,documentation/docs__T_NORM_CLAIM.txt,docs__T_NORM_CLAIM.txt,documentation
49,documentation/docs__T_NORM_CLAIM_ELIGIBILITY_R...,docs__T_NORM_CLAIM_ELIGIBILITY_REVIEW_CROSSWAL...,documentation
50,documentation/docs__T_NORM_CLINICAL_REVIEW.txt,docs__T_NORM_CLINICAL_REVIEW.txt,documentation
